# Multi-Food Detection Exploration

This notebook explores a detector-first path for FoodLens. The current FoodLens model is a **single-label Food-101 classifier**; this notebook tests whether a pretrained detector can propose food regions that can later be classified crop by crop.

The notebook is intentionally separated from the classifier pipeline so detection quality can be reviewed before app integration.

## 1. Install Dependencies And Import Libraries

Kaggle does not include Ultralytics by default, so install it once at the top of the notebook before importing YOLO. Internet must be enabled for this install and for automatic YOLO weight download.


In [ ]:
!pip install -q ultralytics


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw
from ultralytics import YOLO


## 2. Configuration

Set the input mode, attached Kaggle path, detector weights, thresholds, and output locations here. The default `INPUT_PATH` points directly to the attached Kaggle sample folder and processes the supported files inside that directory. `DETECTOR_WEIGHTS = "yolo11n.pt"` downloads weights automatically when internet is enabled; otherwise set it to an attached `.pt` file path.


In [ ]:
@dataclass(frozen=True)
class Config:
    MODE: str = "image"  # image or video
    SEED: int = 42
    INPUT_PATH: Path = Path("/kaggle/input/datasets/tuannm3823/foodlens-demo-samples/foodlens_demo_samples")
    OUTPUT_DIR: Path = Path("/kaggle/working/results/multi_food_detection")
    DETECTOR_WEIGHTS: str = "yolo11n.pt"
    CONFIDENCE_THRESHOLD: float = 0.25
    IOU_THRESHOLD: float = 0.50
    MAX_DETECTIONS: int = 20
    MAX_INPUT_FILES: int = 20
    MAX_VISUAL_INSPECTION_IMAGES: int = 6
    MAX_VISUAL_INSPECTION_CROPS: int = 12
    MIN_CROP_AREA_RATIO: float = 0.015
    MAX_CROP_AREA_RATIO: float = 0.80
    VIDEO_FRAME_STRIDE_SECONDS: float = 2.0
    SUPPORTED_IMAGE_EXTENSIONS: tuple[str, ...] = (
        ".jpg",
        ".jpeg",
        ".png",
        ".webp",
    )
    SUPPORTED_VIDEO_EXTENSIONS: tuple[str, ...] = (
        ".mp4",
        ".mov",
        ".webm",
    )
    CANDIDATE_REGION_LABELS: tuple[str, ...] = (
        "apple",
        "banana",
        "bowl",
        "broccoli",
        "cake",
        "carrot",
        "donut",
        "hot dog",
        "orange",
        "pizza",
        "sandwich",
    )


CFG = Config()
random.seed(CFG.SEED)
np.random.seed(CFG.SEED)
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "crops").mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
CFG

## 3. Load Detector

A pretrained YOLO model is used first as an object-proposal model. Its COCO labels are not Food-101 dish labels, so labels such as `bowl` should be interpreted as candidate crop regions rather than final predictions.

In [ ]:
detector = YOLO(CFG.DETECTOR_WEIGHTS)
detector

## 4. Load Images Or Sampled Video Frames

The input path can be a single file or an exact directory. Image mode processes supported image files directly inside the configured folder. Video mode samples frames at a fixed interval so detection can be tested without processing every frame.


In [ ]:
def supported_extensions() -> tuple[str, ...]:
    """Return supported extensions for the configured input mode."""
    if CFG.MODE == "video":
        return CFG.SUPPORTED_VIDEO_EXTENSIONS
    return CFG.SUPPORTED_IMAGE_EXTENSIONS


def find_input_files(input_path: Path) -> list[Path]:
    """Find supported input files from the configured file or directory.

    Args:
        input_path: Kaggle file or exact dataset directory path.

    Returns:
        Sorted file paths capped by `CFG.MAX_INPUT_FILES`.
    """
    extensions = supported_extensions()
    if input_path.is_file():
        if input_path.suffix.lower() not in extensions:
            raise ValueError(f"Unsupported input file type: {input_path.suffix}")
        return [input_path]

    if not input_path.exists():
        raise FileNotFoundError(f"Input path does not exist: {input_path}")

    input_files = [
        path
        for path in sorted(input_path.iterdir())
        if path.is_file() and path.suffix.lower() in extensions
    ]
    if not input_files:
        raise FileNotFoundError(f"No supported {CFG.MODE} files found in {input_path}")
    return input_files[: CFG.MAX_INPUT_FILES]


def sample_video_frames(
    video_path: Path,
    stride_seconds: float,
) -> list[tuple[int, Image.Image]]:
    """Sample RGB frames from a video file.

    Args:
        video_path: Path to the input video.
        stride_seconds: Seconds between sampled frames.

    Returns:
        A list of `(frame_index, image)` tuples.
    """
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise ValueError(f"Could not open video: {video_path}")

    fps = capture.get(cv2.CAP_PROP_FPS) or 30
    frame_stride = max(1, int(fps * stride_seconds))
    frames: list[tuple[int, Image.Image]] = []
    frame_index = 0

    while True:
        success, frame_bgr = capture.read()
        if not success:
            break
        if frame_index % frame_stride == 0:
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            frames.append((frame_index, Image.fromarray(frame_rgb)))
        frame_index += 1

    capture.release()
    return frames


def load_inputs() -> list[tuple[str, Image.Image]]:
    """Load images or sampled video frames for detection."""
    input_files = find_input_files(CFG.INPUT_PATH)
    inputs: list[tuple[str, Image.Image]] = []

    for input_file in input_files:
        if CFG.MODE == "video":
            sampled_frames = sample_video_frames(
                input_file,
                CFG.VIDEO_FRAME_STRIDE_SECONDS,
            )
            inputs.extend(
                (f"{input_file.stem}_frame_{idx:06d}", image)
                for idx, image in sampled_frames
            )
        else:
            inputs.append((input_file.stem, Image.open(input_file).convert("RGB")))

    return inputs


inputs = load_inputs()
len(inputs)

## 5. Run Detection And Export Crops

The detector exports filtered candidate regions, annotated figures, and individual crops. CSV artifact paths are stored relative to `CFG.OUTPUT_DIR` so downstream notebooks can consume linked Kaggle outputs reliably. Generic context objects such as cups, wine glasses, and dining tables are intentionally excluded from crop export because they are usually not useful Food-101 classification inputs.

In [ ]:
def detector_region_role(detector_label: str) -> str:
    """Map a generic detector label to its FoodLens proposal role."""
    direct_food_labels = {
        "apple",
        "banana",
        "broccoli",
        "cake",
        "carrot",
        "donut",
        "hot dog",
        "orange",
        "pizza",
        "sandwich",
    }
    if detector_label in direct_food_labels:
        return "direct_food"
    if detector_label == "bowl":
        return "serving_container"
    return "context_object"


def should_export_detection(detector_label: str, area_ratio: float) -> bool:
    """Return whether a detector box is useful as a classifier crop."""
    return (
        detector_label in CFG.CANDIDATE_REGION_LABELS
        and CFG.MIN_CROP_AREA_RATIO <= area_ratio <= CFG.MAX_CROP_AREA_RATIO
    )


def draw_proposal_boxes(
    image: Image.Image,
    rows: list[dict[str, object]],
) -> Image.Image:
    """Draw filtered proposal boxes for visual inspection."""
    annotated = image.copy()
    draw = ImageDraw.Draw(annotated)

    for row in rows:
        x1 = int(row["x1"])
        y1 = int(row["y1"])
        x2 = int(row["x2"])
        y2 = int(row["y2"])
        label = str(row["detector_label"])
        confidence = float(row["detector_confidence"])
        role = str(row["proposal_role"])
        color = (213, 0, 32) if role == "direct_food" else (46, 125, 50)
        text = f"{label} {confidence:.2f}"

        draw.rectangle((x1, y1, x2, y2), outline=color, width=4)
        text_box = draw.textbbox((x1, y1), text)
        draw.rectangle(
            (text_box[0] - 4, text_box[1] - 4, text_box[2] + 4, text_box[3] + 4),
            fill=color,
        )
        draw.text((x1, y1), text, fill=(255, 255, 255))

    return annotated


def detect_and_export(
    source_name: str,
    image: Image.Image,
) -> list[dict[str, object]]:
    """Detect candidate food regions and export crop artifacts.

    Args:
        source_name: Stable image or frame identifier.
        image: RGB image to process.

    Returns:
        Filtered detection metadata rows for CSV export.
    """
    result = detector.predict(
        source=np.array(image),
        conf=CFG.CONFIDENCE_THRESHOLD,
        iou=CFG.IOU_THRESHOLD,
        max_det=CFG.MAX_DETECTIONS,
        verbose=False,
    )[0]

    rows: list[dict[str, object]] = []
    boxes = result.boxes
    figure_path = CFG.OUTPUT_DIR / "figures" / f"{source_name}_detections.jpg"
    source_width, source_height = image.size
    source_area = source_width * source_height

    if boxes is not None:
        for detection_index, box in enumerate(boxes):
            x1, y1, x2, y2 = [int(value) for value in box.xyxy[0].tolist()]
            x1 = max(0, min(x1, source_width))
            x2 = max(0, min(x2, source_width))
            y1 = max(0, min(y1, source_height))
            y2 = max(0, min(y2, source_height))
            if x2 <= x1 or y2 <= y1:
                continue

            confidence = float(box.conf[0])
            detector_class_id = int(box.cls[0])
            detector_label = result.names.get(detector_class_id, str(detector_class_id))
            crop_area_ratio = ((x2 - x1) * (y2 - y1)) / source_area
            proposal_role = detector_region_role(detector_label)
            if not should_export_detection(detector_label, crop_area_ratio):
                continue

            crop = image.crop((x1, y1, x2, y2))
            crop_relative_path = Path("crops") / f"{source_name}_crop_{len(rows):02d}.jpg"
            crop_path = CFG.OUTPUT_DIR / crop_relative_path
            crop.save(crop_path, quality=95)

            rows.append(
                {
                    "source_id": source_name,
                    "detection_index": detection_index,
                    "detector_label": detector_label,
                    "proposal_role": proposal_role,
                    "detector_confidence": confidence,
                    "crop_area_ratio": crop_area_ratio,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                    "crop_path": str(crop_relative_path),
                    "figure_path": str(Path("figures") / figure_path.name),
                    "source_width": source_width,
                    "source_height": source_height,
                }
            )

    annotated = draw_proposal_boxes(image, rows)
    annotated.save(figure_path, quality=95)
    return rows


detection_rows: list[dict[str, object]] = []
for source_name, source_image in inputs:
    detection_rows.extend(detect_and_export(source_name, source_image))

detections_df = pd.DataFrame(detection_rows)
detections_df.to_csv(CFG.OUTPUT_DIR / "detections.csv", index=False)
detections_df.head()


## 6. Visual Inspection And Output Review

Review the detection summary, filtered YOLO proposal boxes, and exported crops before moving to classifier-per-crop integration. The key question is whether the detector localizes meaningful food regions cleanly enough for Food-101 classification. If most outputs are containers rather than food regions, the next model improvement should be a food-specific detector or segmentation model.


In [ ]:
def build_detection_summary() -> pd.DataFrame:
    """Build a source-level summary for quick detector review."""
    rows: list[dict[str, object]] = []

    for source_name, _ in inputs:
        if detections_df.empty:
            source_detections = pd.DataFrame()
        else:
            source_detections = detections_df[detections_df["source_id"] == source_name]

        if source_detections.empty:
            rows.append(
                {
                    "source_id": source_name,
                    "detection_count": 0,
                    "top_detector_label": None,
                    "top_detector_confidence": None,
                    "mean_detector_confidence": None,
                    "proposal_roles": None,
                }
            )
            continue

        top_detection = source_detections.loc[
            source_detections["detector_confidence"].idxmax()
        ]
        rows.append(
            {
                "source_id": source_name,
                "detection_count": len(source_detections),
                "top_detector_label": top_detection["detector_label"],
                "top_detector_confidence": top_detection["detector_confidence"],
                "mean_detector_confidence": source_detections[
                    "detector_confidence"
                ].mean(),
                "proposal_roles": ", ".join(
                    sorted(source_detections["proposal_role"].unique())
                ),
            }
        )

    return pd.DataFrame(rows)


def show_annotated_detections(
    figure_dir: Path,
    limit: int = CFG.MAX_VISUAL_INSPECTION_IMAGES,
    columns: int = 2,
) -> None:
    """Display YOLO-annotated images with bounding boxes."""
    figure_paths = sorted(figure_dir.glob("*_detections.jpg"))[:limit]
    if not figure_paths:
        print(f"No annotated detection images found in {figure_dir}")
        return

    columns = min(columns, len(figure_paths))
    rows = int(np.ceil(len(figure_paths) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(6 * columns, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for axis, figure_path in zip(axes, figure_paths):
        axis.imshow(Image.open(figure_path))
        axis.set_title(figure_path.stem.replace("_detections", ""), fontsize=10)
        axis.axis("off")

    for axis in axes[len(figure_paths):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()


def show_detection_crops(
    detections: pd.DataFrame,
    limit: int = CFG.MAX_VISUAL_INSPECTION_CROPS,
    columns: int = 4,
) -> None:
    """Display the highest-confidence exported detector crops."""
    if detections.empty:
        print("No detector crops were exported.")
        return

    crop_rows = detections.sort_values(
        "detector_confidence",
        ascending=False,
    ).head(limit)
    columns = min(columns, len(crop_rows))
    rows = int(np.ceil(len(crop_rows) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(3.2 * columns, 3.2 * rows))
    axes = np.array(axes).reshape(-1)

    for axis, (_, row) in zip(axes, crop_rows.iterrows()):
        crop_path = Path(row["crop_path"])
        if not crop_path.is_absolute():
            crop_path = CFG.OUTPUT_DIR / crop_path
        axis.imshow(Image.open(crop_path))
        title = f"{row['detector_label']} | {row['detector_confidence']:.2f}"
        axis.set_title(title, fontsize=9)
        axis.axis("off")

    for axis in axes[len(crop_rows):]:
        axis.axis("off")

    plt.tight_layout()
    plt.show()


summary_df = build_detection_summary()
summary_df.to_csv(CFG.OUTPUT_DIR / "detection_summary.csv", index=False)
display(summary_df)
show_annotated_detections(CFG.OUTPUT_DIR / "figures")
show_detection_crops(detections_df)
